# OMNI MOVIE STUDIO — FREE GPU BRIDGE v3 (Colab T4)
Setup ~15 min (models download once). **Keep this tab open** — closing it disconnects the free GPU.
Runtime → Run all. Wait for the last cell to print `ALL SERVICES UP` + two URLs, then paste them to your agent.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
import os
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/omni-movie'
os.makedirs(OUT, exist_ok=True)
print('drive ready:', OUT)

In [ ]:
# 1) ComfyUI (starts FIRST, downloads models next cell)
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI > /dev/null 2>&1
%pip -q install -r /content/ComfyUI/requirements.txt
%cd /content/ComfyUI
import subprocess, threading, time, urllib.request
threading.Thread(target=lambda: subprocess.run(['python','main.py','--port','8188','--listen','0.0.0.0','--dont-print-server']), daemon=True).start()
print('comfyui thread started (models download in next cell...)')

In [ ]:
# 2) MODELS — SDXL (stills) + Wan 2.1 I2V 480p fp8 (animation). Free open weights, ~18GB.
M='https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files'
!wget -q -P /content/ComfyUI/models/unet {M}/diffusion_models/wan2.1_i2v_480p_9.5B_fp8_e4m3fn.safetensors
!wget -q -P /content/ComfyUI/models/text_encoders {M}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
!wget -q -P /content/ComfyUI/models/clip_vision {M}/clip_vision/clip_vision_h.safetensors
!wget -q -P /content/ComfyUI/models/vae {M}/vae/wan_2.1_vae.safetensors
!wget -q -P /content/ComfyUI/models/checkpoints https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors
import subprocess
for d in ['unet','text_encoders','clip_vision','vae','checkpoints']:
    print(d, ':', subprocess.run(['ls','-lh',f'/content/ComfyUI/models/{d}'],capture_output=True,text=True).stdout.strip().split('\n')[-1])
print('MODELS DOWNLOADED')

In [ ]:
# 3) Wav2Lip — free lip-sync
!git clone --depth 1 https://github.com/Rudrabha/Wav2Lip /content/Wav2Lip > /dev/null 2>&1
%cd /content/Wav2Lip
!mkdir -p checkpoints face_detection/detection/sfd
!wget -q https://github.com/justinjohn0306/Wav2Lip/releases/download/Models/wav2lip_gan.pth -O checkpoints/wav2lip_gan.pth || echo 'wav2lip weights: get from Wav2Lip repo releases'
!wget -q https://github.com/justinjohn0306/Wav2Lip/releases/download/Models/s3fd.pth -O face_detection/detection/sfd/s3fd.pth || echo 's3fd: get from repo'
%pip -q install librosa==0.10.1 numba==0.58.1
print('wav2lip ready (env: LIPSYNC_ENGINE=wav2lip)')

In [ ]:
# 4) XTTS v2 — expressive Hindi TTS server on :8020
%pip -q install TTS==0.22.0
server = '''
from TTS.api import TTS
from http.server import BaseHTTPRequestHandler, HTTPServer
import json
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
class H(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/health':
            self.send_response(200); self.end_headers(); self.wfile.write(b'ok')
    def do_POST(self):
        n = int(self.headers.get('content-length', 0)); body = json.loads(self.rfile.read(n))
        out = body.get('out', '/tmp/x.wav')
        tts.tts_to_file(text=body['text'], language=body.get('language','hi'), file_path=out)
        self.send_response(200); self.send_header('content-type','audio/wav'); self.end_headers()
        self.wfile.write(open(out,'rb').read())
HTTPServer(('0.0.0.0', 8020), H).serve_forever()
'''
open('/content/xtts_server.py','w').write(server)
import threading, subprocess
threading.Thread(target=lambda: subprocess.run(['python','/content/xtts_server.py']), daemon=True).start()
print('xtts thread started (model loading...)')

In [ ]:
# 5) VERIFY + PUBLIC URLS — waits until everything is truly UP, then prints
import time, urllib.request
def wait_up(url, timeout_s, label):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            urllib.request.urlopen(url, timeout=5); return True
        except Exception: time.sleep(10)
    return False
ok1 = wait_up('http://127.0.0.1:8188/system_stats', 1800, 'comfyui')
ok2 = wait_up('http://127.0.0.1:8020/health', 1800, 'xtts')
print('comfyui up:', ok1, '| xtts up:', ok2)
if not (ok1 and ok2):
    raise SystemExit('A service failed to start — scroll up for errors and rerun this cell.')
from google.colab.output import eval_js
comfy_url = eval_js('google.colab.kernel.proxyPort(8188)')
xtts_url = eval_js('google.colab.kernel.proxyPort(8020)')
print('='*60)
print('ALL SERVICES UP — PASTE THESE TO YOUR AGENT (keep private, keep tab open):')
print(f'COMFYUI_URL={comfy_url}')
print(f'XTTS_SERVER_URL={xtts_url}')
print('='*60)